# 03 — End to end: dbt project on Delta + Iceberg, PIT queries, PIT analysis

**The scenario.** You have an analytics project: a dbt model `revenue_summary`
that joins a Delta Lake fact table (`orders`) with an Iceberg dimension
(`customers`). Retention jobs have destroyed part of both tables' history.
You want to run the model *as of a past date* — and know whether the answer
can be trusted.

**What alethe does here:**
1. Derives an empirically-validated watermark for each physical table
2. Walks the dbt DAG to find what `revenue_summary` depends on
3. Produces the PIT achievability report (CERTAIN / BOUNDED / UNACHIEVABLE)
4. Rewrites the model's compiled SQL with engine-native time travel — and
   *refuses* when the requested time is unanswerable
5. Proves the rewrite is honest with real time-travel reads

**Install:**
```bash
pip install -e ".[all]"      # from the repo root — or: pip install alethe[all]
```

In [1]:
import alethe
print(f"alethe {alethe.__version__}")

alethe 0.1.0


---
## 1. The project

We build a self-contained workspace under `./e2e_workspace/`:

```
e2e_workspace/
├── lakehouse/orders          ← Delta table, 6 versions, then VACUUM
├── iceberg/                  ← Iceberg warehouse, raw.customers, history destroyed
└── dbt/target/
    ├── manifest.json         ← what `dbt compile` produces
    ├── run_results.json      ← what `dbt run` produces
    └── compiled/analytics/revenue_summary.sql
```

In a real project the dbt artifacts already exist — you'd point alethe at
`target/`. Here we generate them so the notebook is fully reproducible.

In [2]:
import json, shutil, time
from pathlib import Path
from datetime import datetime, timezone, timedelta

WS = Path("e2e_workspace")
if WS.exists():
    shutil.rmtree(WS)
(WS / "dbt" / "target" / "compiled" / "analytics").mkdir(parents=True)
print(f"workspace: {WS.resolve()}")

workspace: /Users/seamusaran/Documents/alethe/notebooks/e2e_workspace


### 1a. Delta fact table — write history, then destroy it

Six versions of `orders`, written seconds apart so the retention window is
visible, then a real `VACUUM` with zero retention — the footgun every engine
exposes and OWS exists to make honest.

In [3]:
import pandas as pd
from deltalake import DeltaTable, write_deltalake

DELTA_ORDERS = WS / "lakehouse" / "orders"

for v in range(6):
    df = pd.DataFrame({
        "order_id":    [f"O{v}-{i}" for i in range(5)],
        "customer_id": [f"C{i % 3}" for i in range(5)],
        "amount":      [100.0 * (v + 1) + i for i in range(5)],
    })
    write_deltalake(DELTA_ORDERS, df, mode="overwrite")
    time.sleep(1)   # keep commit timestamps distinct

removed = DeltaTable(str(DELTA_ORDERS)).vacuum(
    retention_hours=0, enforce_retention_duration=False, dry_run=False)

listed = [h["version"] for h in DeltaTable(str(DELTA_ORDERS)).history()
          if h.get("operation") == "WRITE"]
print(f"6 versions written, {len(removed)} parquet files vacuumed")
print(f"_delta_log still lists write versions: {sorted(listed)}  ← the lie")

6 versions written, 5 parquet files vacuumed
_delta_log still lists write versions: [0, 1, 2, 3, 4, 5]  ← the lie


### 1b. Iceberg dimension — same story, different metadata model

In [4]:
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

ICE_WH = WS / "iceberg"
ICE_WH.mkdir()
catalog = SqlCatalog("local",
                     uri=f"sqlite:///{ICE_WH}/catalog.db",
                     warehouse=f"file://{ICE_WH.resolve()}")
catalog.create_namespace("raw")
cust = catalog.create_table("raw.customers", pa.schema([
    ("customer_id", pa.string()), ("segment", pa.string())]))

for s in range(4):
    cust.overwrite(pa.table({
        "customer_id": ["C0", "C1", "C2"],
        "segment": ["smb", "enterprise", "smb"]}))
    time.sleep(1)

# Destroy old data files out-of-band (lifecycle policy / cleanup script)
snaps = sorted(cust.metadata.snapshots, key=lambda s: s.sequence_number)
keep = {t.file.file_path for sn in snaps[-2:]
        for t in cust.scan(snapshot_id=sn.snapshot_id).plan_files()}
destroyed = 0
for sn in snaps[:-2]:
    try:
        for t in cust.scan(snapshot_id=sn.snapshot_id).plan_files():
            p = Path(t.file.file_path.replace("file://", ""))
            if t.file.file_path not in keep and p.exists():
                p.unlink(); destroyed += 1
    except Exception:
        pass
print(f"{len(snaps)} snapshots in metadata, {destroyed} data files destroyed out-of-band")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


7 snapshots in metadata, 3 data files destroyed out-of-band


### 1c. The dbt artifacts

`manifest.json` encodes the DAG that `{{ source() }}` and `{{ ref() }}` build:

```
source: raw.orders (Delta)     source: raw.customers (Iceberg)
        │                              │
  stg_orders ──────────┬─── stg_customers
                revenue_summary
```

In [5]:
TARGET = WS / "dbt" / "target"

manifest = {
  "metadata": {"dbt_schema_version": "https://schemas.getdbt.com/dbt/manifest/v10/manifest.json"},
  "nodes": {
    "model.analytics.revenue_summary": {
      "unique_id": "model.analytics.revenue_summary",
      "name": "revenue_summary", "resource_type": "model",
      "path": "revenue_summary.sql",
      "depends_on": {"nodes": ["model.analytics.stg_orders",
                               "model.analytics.stg_customers"]}},
    "model.analytics.stg_orders": {
      "unique_id": "model.analytics.stg_orders",
      "name": "stg_orders", "resource_type": "model",
      "depends_on": {"nodes": ["source.analytics.raw.orders"]}},
    "model.analytics.stg_customers": {
      "unique_id": "model.analytics.stg_customers",
      "name": "stg_customers", "resource_type": "model",
      "depends_on": {"nodes": ["source.analytics.raw.customers"]}},
  },
  "sources": {
    "source.analytics.raw.orders": {
      "unique_id": "source.analytics.raw.orders", "resource_type": "source",
      "source_name": "raw", "name": "orders", "schema": "raw",
      "database": "lake", "relation_name": "lake.raw.orders"},
    "source.analytics.raw.customers": {
      "unique_id": "source.analytics.raw.customers", "resource_type": "source",
      "source_name": "raw", "name": "customers", "schema": "raw",
      "database": "lake", "relation_name": "lake.raw.customers"},
  },
}
(TARGET / "manifest.json").write_text(json.dumps(manifest, indent=2))

# The compiled model — what dbt renders after resolving {{ source() }} / {{ ref() }}.
# stg CTEs inlined, exactly as an ephemeral materialization would produce.
compiled_sql = """\
with stg_orders as (
    select order_id, customer_id, amount
    from lake.raw.orders
    where amount is not null
),
stg_customers as (
    select customer_id, segment
    from lake.raw.customers
)
select
    c.segment,
    count(*)        as n_orders,
    sum(o.amount)   as revenue
from stg_orders o
join stg_customers c on o.customer_id = c.customer_id
group by c.segment
"""
(TARGET / "compiled" / "analytics" / "revenue_summary.sql").write_text(compiled_sql)

# run_results.json — the model was materialized just now (after the vacuum)
now_iso = datetime.now(tz=timezone.utc).isoformat()
(TARGET / "run_results.json").write_text(json.dumps({
  "results": [{
    "unique_id": "model.analytics.revenue_summary",
    "status": "success",
    "timing": [{"name": "execute",
                "started_at": now_iso, "completed_at": now_iso}]}]
}))
print("dbt artifacts written to", TARGET)

dbt artifacts written to e2e_workspace/dbt/target


---
## 2. Point alethe at the physical tables

One call per table. Each watermark is **empirically validated**: alethe
performs a real read at the boundary (must succeed) and below it (must fail)
— metadata arithmetic is never trusted alone.

In [6]:
wm_orders    = alethe.watermark(DELTA_ORDERS)   # auto-detects Delta
wm_customers = alethe.watermark("raw.customers", adapter="iceberg", catalog=catalog)

for wm in (wm_orders, wm_customers):
    print(wm.chain)
    print(f"   boundary:   {wm.boundary}")
    print(f"   boundary_dt: {wm.boundary_dt.strftime('%H:%M:%S')}   "
          f"earliest_dt: {wm.earliest_dt.strftime('%H:%M:%S')}")
    print(f"   grade: {wm.evidence_grade.value}   "
          f"empirically_validated: {wm.empirically_validated}")

# Record both claims in the hash-chained manifest
for wm in (wm_orders, wm_customers):
    alethe.record(wm, WS / "watermarks.jsonl")
print()
print(alethe.Manifest(WS / "watermarks.jsonl"))

delta://orders
   boundary:   {'version': 5}
   boundary_dt: 15:23:41   earliest_dt: 15:23:36
   grade: derived   empirically_validated: True
iceberg://raw.customers
   boundary:   {'snapshot_id': '7029050243041390726', 'sequence_number': 6, 'timestamp_ms': 1783178625981}
   boundary_dt: 15:23:45   earliest_dt: 15:23:42
   grade: derived   empirically_validated: True

Manifest('watermarks.jsonl', 2 entries, INTACT)


---
## 3. The PIT analysis report

`DbtLineage` walks the DAG from `revenue_summary` down to its sources,
composes their watermarks with weakest-link semantics, and applies the
twice-temporal correction from `run_results.json`.

In [7]:
from alethe.integrations import DbtLineage

lineage = DbtLineage(TARGET / "manifest.json")
print(lineage)
print("upstream leaves of revenue_summary:",
      [n["unique_id"] for n in lineage.upstream_leaves("revenue_summary")])

watermarks = {
    "source.analytics.raw.orders":    wm_orders,
    "source.analytics.raw.customers": wm_customers,
}

report = lineage.pit_report(
    "revenue_summary",
    watermarks=watermarks,
    run_results_path=TARGET / "run_results.json",
)
print()
print(report)

DbtLineage('manifest.json', 3 models, 2 leaf nodes, schema='https://schemas.getdbt.com/dbt/manifest/v10/manifest.json')
upstream leaves of revenue_summary: ['source.analytics.raw.orders', 'source.analytics.raw.customers']

PIT Achievability Report: revenue_summary
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://orders                     2026-07-04 15:23:41      EvidenceGrade.DERIVED
  iceberg://raw.customers            2026-07-04 15:23:45      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:23:45  (limiting: iceberg://raw.customers)
Effective grade:     EvidenceGrade.DERIVED
Materialization:     2026-07-04 15:23:47  [twice-temporal: CONFORMANT]

PIT zones:
  CERTAIN        2026-07-04 15:23:45 → now
  BOUNDED        2026-07-04 15:23:42 → 2026-07-04 15:23:45  limiting: ['iceberg://raw.customers']
  UNACHIEVABLE   −∞ → 2026-07

---
## 4. Point-in-time queries — three zones, three outcomes

`rewrite_model()` binds every source reference in the compiled SQL to the
requested time using the engine's native time-travel syntax. The PIT report
gates it: an unanswerable time is **refused**, not silently emptied.

In [8]:
# ── CERTAIN: now — both sources fully retained ──────────────────────
t_certain = datetime.now(tz=timezone.utc)

res = lineage.rewrite_model(
    "revenue_summary", t_certain,
    watermarks=watermarks,
    compiled_path=TARGET / "compiled" / "analytics",
    dialect="spark",
)
print(f"zone: {res.status.value}   bound: {res.bound_tables}")
print()
print(res.sql)

zone: CERTAIN   bound: ['lake.raw.orders', 'lake.raw.customers']

WITH stg_orders AS (
  SELECT
    order_id,
    customer_id,
    amount
  FROM lake.raw.orders TIMESTAMP AS OF CAST('2026-07-04 15:23:47' AS TIMESTAMP)
  WHERE
    NOT amount IS NULL
), stg_customers AS (
  SELECT
    customer_id,
    segment
  FROM lake.raw.customers TIMESTAMP AS OF CAST('2026-07-04 15:23:47' AS TIMESTAMP)
)
SELECT
  c.segment,
  COUNT(*) AS n_orders,
  SUM(o.amount) AS revenue
FROM stg_orders AS o
JOIN stg_customers AS c
  ON o.customer_id = c.customer_id
GROUP BY
  c.segment


In [9]:
# Same query, Trino dialect — for an Iceberg-native engine
res_trino = lineage.rewrite_model(
    "revenue_summary", t_certain,
    watermarks=watermarks,
    compiled_path=TARGET / "compiled" / "analytics",
    dialect="trino",
)
print(res_trino.sql.split("WITH")[0], "...")
for line in res_trino.sql.splitlines():
    if "FOR TIMESTAMP" in line:
        print(line.strip())

 ...
FROM lake.raw.orders FOR TIMESTAMP AS OF CAST('2026-07-04 15:23:47' AS TIMESTAMP)
FROM lake.raw.customers FOR TIMESTAMP AS OF CAST('2026-07-04 15:23:47' AS TIMESTAMP)


In [10]:
# ── BOUNDED: between table creation and the retention boundary ──────
t_bounded = report.earliest_dt + (report.effective_boundary - report.earliest_dt) / 2

res_b = lineage.rewrite_model(
    "revenue_summary", t_bounded,
    watermarks=watermarks,
    compiled_path=TARGET / "compiled" / "analytics",
)
print(f"zone: {res_b.status.value}")
for w in res_b.warnings:
    print(f"⚠  {w}")

zone: BOUNDED
⚠  AS OF 2026-07-04T15:23:44.440500+00:00 is inside the BOUNDED zone: retention has destroyed part of ['iceberg://raw.customers']. Monotone aggregates are lower bounds; non-monotone queries (NOT EXISTS, MIN/MAX) should be REFUSED.


In [11]:
# ── UNACHIEVABLE: before the tables existed — honest refusal ────────
from alethe.integrations import UnachievableQueryError

t_impossible = report.earliest_dt - timedelta(days=30)
try:
    lineage.rewrite_model(
        "revenue_summary", t_impossible,
        watermarks=watermarks,
        compiled_path=TARGET / "compiled" / "analytics",
    )
except UnachievableQueryError as e:
    print(f"REFUSED: {e}")

REFUSED: AS OF 2026-06-04T15:23:42.900000+00:00 precedes the existence of ['iceberg://raw.customers']. No honest rewrite is possible — the population itself is unknowable at that time (spec §9).


---
## 5. Prove the rewrite is honest — real time-travel reads

The rewritten SQL claims the sources are readable at `t_certain`. Alethe's
conformance rule says: validate empirically, never trust metadata. So we
execute the same point-in-time reads the rewritten query would perform —
at the boundary (must succeed) and below it (must fail).

In [12]:
# Delta: read AS OF the boundary version — the rewrite's target state
boundary_v = wm_orders.boundary["version"]
dt = DeltaTable(str(DELTA_ORDERS), version=boundary_v)
df_pit = dt.to_pyarrow_table().to_pandas()
print(f"read AS OF v{boundary_v}: {len(df_pit)} rows — the PIT query would see:")
print(df_pit.head(3).to_string(index=False))

# And below the boundary: the read the rewrite REFUSES to attempt
try:
    DeltaTable(str(DELTA_ORDERS), version=boundary_v - 1).to_pyarrow_table()
    print("\nv{boundary_v-1}: readable (unexpected!)")
except Exception as e:
    print(f"\nread AS OF v{boundary_v - 1}: FAILS ({type(e).__name__})"
          f"  ← this is what a naive engine hits at runtime with no warning")

read AS OF v5: 5 rows — the PIT query would see:
order_id customer_id  amount
    O5-0          C0   600.0
    O5-1          C1   601.0
    O5-2          C2   602.0

read AS OF v4: FAILS (FileNotFoundError)  ← this is what a naive engine hits at runtime with no warning


In [13]:
# Iceberg: scan at the boundary snapshot vs a destroyed one
b_snap = int(wm_customers.boundary["snapshot_id"])
n = cust.scan(snapshot_id=b_snap).to_arrow().num_rows
print(f"scan at boundary snapshot {str(b_snap)[:8]}…: {n} rows — readable")

destroyed_snaps = [s for s in snaps
                   if str(s.snapshot_id) not in
                   [str(b_snap)] + [i["snapshot_id"] for i in wm_customers.readable_islands]
                   and s.sequence_number < wm_customers.boundary["sequence_number"]]
if destroyed_snaps:
    sn = destroyed_snaps[0]
    try:
        cust.scan(snapshot_id=sn.snapshot_id).to_arrow()
        print(f"scan at {str(sn.snapshot_id)[:8]}…: readable (island)")
    except Exception as e:
        print(f"scan at destroyed snapshot {str(sn.snapshot_id)[:8]}…: "
              f"FAILS ({type(e).__name__}) — metadata listed it anyway")

scan at boundary snapshot 70290502…: 0 rows — readable
scan at destroyed snapshot 68418867…: FAILS (FileNotFoundError) — metadata listed it anyway


---
## 6. Recap

| Step | Call | Output |
|---|---|---|
| Watermark the tables | `alethe.watermark(path)` | boundary + evidence grade, empirically validated |
| Record the claims | `alethe.record(wm, manifest)` | hash-chained, tamper-evident ledger |
| Analyse the model | `lineage.pit_report(model, ...)` | CERTAIN / BOUNDED / UNACHIEVABLE zones + twice-temporal check |
| Query a point in time | `lineage.rewrite_model(model, since, ...)` | engine-native time-travel SQL — or an honest refusal |

The thing to notice: **the refusal is a feature.** A naive `AS OF` against
vacuumed history either errors at runtime or — worse — silently returns
empty results that look like real zeros. Alethe converts that into a
plan-time verdict with the limiting table named.